In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bhuvant1890/route-resilience-unet-r34/checkpoints/unet_r34/best.pt
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/.gitignore
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/Makefile
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/README.md
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/docker-compose.yml
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/pyproject.toml
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/requirements-dev.txt
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/requirements-dl.txt
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/requirements-geo.txt
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/requirements.txt
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/backend/README.md
/kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai/backend/app/__init__.py
/kaggle/input/dataset

In [2]:
RUN = "segformer"

In [3]:
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/scripts/kaggle_train.py', recursive=True)
assert hits, "project dataset attach nahi hai"
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
sat = glob.glob('/kaggle/input/**/train/*_sat.jpg', recursive=True)
assert sat, "DeepGlobe dataset attach nahi hai"
DEEPGLOBE = os.path.dirname(os.path.dirname(sat[0]))
shutil.copytree(PROJECT, '/kaggle/working/rr', dirs_exist_ok=True)
os.chdir('/kaggle/working/rr')
print("RUN      :", RUN); print("PROJECT  :", PROJECT); print("DEEPGLOBE:", DEEPGLOBE)
!pip -q install segmentation-models-pytorch
!nvidia-smi -L

RUN      : segformer
PROJECT  : /kaggle/input/datasets/bhuvant1890/routeres/route-resilience-ai
DEEPGLOBE: /kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset
GPU 0: Tesla T4 (UUID: GPU-c9b52c91-0ef3-24b3-e8d2-af81f089c870)
GPU 1: Tesla T4 (UUID: GPU-2271aaf1-59f2-01fd-86c0-03e2d6580bb1)


In [4]:
!python scripts/prepare_deepglobe.py --src "$DEEPGLOBE" --out /kaggle/working/processed

  val    747 tiles
  test   498 tiles
  train  4981 tiles

Wrote /kaggle/working/processed/manifest.json

Next:
  python -m src.models.train --model unet_r34 --data data/processed --epochs 40
  python -m src.models.train --model segformer_b3 --data data/processed --epochs 40
  python -m src.models.train --model segformer_b3 --data data/processed --no-occlusion


In [5]:
D = "/kaggle/working/processed"
if RUN == "deeplab":
    OUT = "/kaggle/working/out/deeplabv3plus_r50"
    !python scripts/kaggle_train.py --data $D --arch deeplabv3plus_r50 --out $OUT --epochs 50 --max-hours 7.5
elif RUN == "segformer":
    OUT = "/kaggle/working/out/segformer_b3"
    !python scripts/kaggle_train.py --data $D --arch segformer_b3 --lr 6e-5 --out $OUT --max-hours 6
elif RUN == "ablation":
    OUT = "/kaggle/working/out/unet_r34_noocc"
    !python scripts/kaggle_train.py --data $D --arch unet_r34 --no-occlusion --out $OUT --max-hours 7
elif RUN == "eval_unet":
    OUT = "/kaggle/working/out/unet_r34"
    os.makedirs(OUT, exist_ok=True)
    shutil.copy(glob.glob('/kaggle/input/**/best.pt', recursive=True)[0], OUT + "/best.pt")

  Route Resilience - segformer_b3 / DeepGlobe training

[1/7] Dependencies
  OK    segmentation-models-pytorch importable   [0.6.0]
  OK    torch available   [2.11.0+cu130]
  OK    CUDA device present   [Tesla T4]

[2/7] Dataset structure
  OK    data root exists   [/kaggle/working/processed]
  OK    train/images and train/masks exist
  OK    val/images and val/masks exist
  OK    test/images and test/masks exist

[3/7] Counts
        train  images=4981   masks=4981   pairs=4981
        val    images=747    masks=747    pairs=747
        test   images=498    masks=498    pairs=498
  OK    train split is non-empty   [4981 pairs]
  OK    val split is non-empty   [747 pairs]

[4/7] Pairing
  OK    every image has a mask   [0 unpaired: []]
  OK    image and mask dimensions agree   [0 mismatched: []]

[5/7] Mask values
  OK    masks are binary   [0 non-binary: []]
  OK    road pixel fraction is plausible   [0.0429 (DeepGlobe is typically 0.02-0.08)]
        mean road fraction 0.0429 -> back

In [6]:
shutil.make_archive(f"/kaggle/working/{RUN}_model", "zip", OUT)
!python scripts/evaluate_checkpoint.py --checkpoint $OUT/best.pt --data /kaggle/working/processed --out $OUT/report --skip-occlusion --skip-classical
shutil.make_archive(f"/kaggle/working/{RUN}_package", "zip", OUT)
shutil.rmtree("/kaggle/working/processed", ignore_errors=True)
print("DONE")

  Post-training evaluation
  checkpoint : /kaggle/working/out/segformer_b3/best.pt
  arch       : segformer_b3   epoch 35   best cldice=0.68153
  device     : cuda   threshold 0.5   tile 512
  git        : None

[val] 747 tiles
       IoU 0.62775  Dice 0.76104  P 0.69854  R 0.86517  clDice 0.85076  PixAcc 0.97747

[test] 498 tiles
       IoU 0.64408  Dice 0.77322  P 0.7059  R 0.88249  clDice 0.85624  PixAcc 0.97881

  val - test IoU gap: -0.01633  (consistent)

Wrote /kaggle/working/out/segformer_b3/report/evaluation_report.json and .md

# Evaluation report

- checkpoint: `/kaggle/working/out/segformer_b3/best.pt`
- architecture: `segformer_b3`  |  epoch 35  |  selected on **cldice** = 0.68153
- threshold 0.5  |  tile 512  |  seed 1234

## 1-2. Validation and held-out test

| split | n | iou | dice | precision | recall | cldice | pixel_accuracy |
|---|---|---|---|---|---|---|---|
| val | 747 | 0.6278 | 0.7610 | 0.6985 | 0.8652 | 0.8508 | 0.9775 |
| test | 498 | 0.6441 | 0.7732 | 0.7059